# Task 4 — EKF · REKF · KalmanNet · existing robust KalmanNet · proposed

Compares the five estimators on the **Synthetic Non-Linear model (Sec IV-C)**
under **full information**, reporting **posterior state-MSE in dB** on a shared
test set. See `docs/ORIGINAL_VS_MODIFIED.md` for what distinguishes *existing*
(professor's original, run via the frozen `*_original` pair) from *proposed*
(our GRU + BPTT version).

Set `CONFIG = 'SMOKE'` to verify quickly, `'FULL'` for the reported numbers.
Partial-information (model-mismatch) is a separate build (needs the widened
`c_range`).

In [1]:
import math, time, random
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

import Simulations.config as config
from Simulations.Synthetic_NL_model.parameters import (
    Q_structure, R_structure, m, n, m1x_0, m2x_0, f, h)
from Simulations.Extended_sysmdl import SystemModel
from Simulations.utils import DataGen
from Filters.EKF_test import EKFTest
from RobustKalmanPY.robust_kalman import RobustKalman                              # proposed (GRU + BPTT)
from RobustKalmanPY.robust_kalman_original import RobustKalman as RobustKalmanOrig # existing (frozen original)
from Pipelines.Pipeline_EKF import Pipeline_EKF
from KNet.KalmanNet_nn import KalmanNetNN

crit = nn.MSELoss()
def to_dB(x): return 10.0 * math.log10(x)

/opt/homebrew/Caskroom/miniconda/base/envs/kalman_net/lib/python3.9/site-packages/torch/torch_version.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]
/opt/homebrew/Caskroom/miniconda/base/envs/kalman_net/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
CONFIG = 'FULL'   # 'SMOKE' (fast check) or 'FULL' (reported numbers)

args = config.general_settings()
args.use_cuda = False; args.randomLength = False
args.lr, args.wd = 1e-3, 1e-3
device = torch.device('cpu')
path_results = 'KNet/'
DatafolderName = 'Simulations/Synthetic_NL_model/data/'
dataFileName = ['data_synNL_T100.pt']

if CONFIG == 'FULL':
    args.N_E, args.N_CV, args.N_T = 200, 50, 100
    args.T, args.T_test = 100, 100
    args.n_batch, args.n_steps = 10, 200
    num_of_test_sets = 100
else:  # SMOKE
    args.N_E, args.N_CV, args.N_T = 10, 5, 5
    args.T, args.T_test = 50, 50
    args.n_batch, args.n_steps = 3, 5
    num_of_test_sets = 3

# Noise: v = q2/r2 = -20 dB, R = 0.1 I  (Sec IV-C)
r2 = torch.tensor([0.1]); v = 10 ** (-20 / 10); q2 = torch.mul(v, r2)
Q = q2[0] * Q_structure; R = r2[0] * R_structure
print(f'CONFIG={CONFIG}  N_E={args.N_E} N_CV={args.N_CV} N_T={args.N_T} '
      f'T={args.T} epochs={args.n_steps} n_batch={args.n_batch} test_sets={num_of_test_sets}')

CONFIG=FULL  N_E=200 N_CV=50 N_T=100 T=100 epochs=200 n_batch=10 test_sets=100


## System model + data (shared across all methods)

In [3]:
sys_model = SystemModel(f, Q, h, R, args.T, args.T_test, m, n)
sys_model.InitSequence(m1x_0, m2x_0)
# attributes used by the hard-coded Jacobian option
sys_model.alpha, sys_model.beta, sys_model.phi, sys_model.delta = 0.9, 1.1, 0.1*math.pi, 0.01
sys_model.a, sys_model.b, sys_model.c = 1, 1, 0

# training / CV pool
DataGen(args, sys_model, DatafolderName + dataFileName[0])
[train_input, train_target, cv_input, cv_target, _, _, _, _, _] = torch.load(
    DatafolderName + dataFileName[0], map_location=device)

# shared test set (list of batches), reused by every method
test_inputs, test_targets = [], []
for _ in range(num_of_test_sets):
    DataGen(args, sys_model, DatafolderName + dataFileName[0])
    [_, _, _, _, ti, tt, _, _, _] = torch.load(DatafolderName + dataFileName[0], map_location=device)
    test_inputs.append(ti); test_targets.append(tt)

# every filter starts from the same initial condition
sys_model.m1x_0 = torch.zeros(m, 1)
c = 1e-3
print('train', tuple(train_input.shape), ' test batches', num_of_test_sets,
      'x', tuple(test_inputs[0].shape))

train (200, 2, 100)  test batches 100 x (100, 2, 100)


## Metric

Every method is scored identically: per-sequence **posterior** (`x_{t|t}`) MSE
over the same test set, then `10·log10` of the mean.

In [4]:
results = {}

def record(name, mses, times):
    mse_lin = float(np.mean(mses))
    mdB = to_dB(mse_lin)
    sdB = float(np.std([to_dB(x) for x in mses]))
    tavg = float(np.mean(times))
    results[name] = {'MSE [dB]': mdB, 'STD [dB]': sdB, 't/seq [s]': tavg}
    print(f'{name:16s}  MSE(dB)={mdB:+8.3f}  STD(dB)={sdB:6.3f}  t/seq={tavg*1e3:7.2f} ms  (nseq={len(mses)})')

## Full-information comparison

### 1. EKF

In [5]:
mses, times = [], []
for ti, tt in zip(test_inputs, test_targets):
    t0 = time.time(); out = EKFTest(args, sys_model, ti, tt); dt = time.time() - t0
    mses.extend(out[0].tolist()); times.extend([dt / ti.shape[0]] * ti.shape[0])
record('EKF', mses, times)

Extended Kalman Filter - MSE LOSS: tensor(-22.2112) [dB]
Extended Kalman Filter - STD: tensor(0.2669) [dB]
Inference Time: 0.964846134185791
Extended Kalman Filter - MSE LOSS: tensor(-22.2223) [dB]
Extended Kalman Filter - STD: tensor(0.2583) [dB]
Inference Time: 0.9746041297912598
Extended Kalman Filter - MSE LOSS: tensor(-22.2206) [dB]
Extended Kalman Filter - STD: tensor(0.2684) [dB]
Inference Time: 1.0006780624389648
Extended Kalman Filter - MSE LOSS: tensor(-22.1647) [dB]
Extended Kalman Filter - STD: tensor(0.2454) [dB]
Inference Time: 0.9828929901123047
Extended Kalman Filter - MSE LOSS: tensor(-22.2205) [dB]
Extended Kalman Filter - STD: tensor(0.2976) [dB]
Inference Time: 0.9624729156494141
Extended Kalman Filter - MSE LOSS: tensor(-22.2052) [dB]
Extended Kalman Filter - STD: tensor(0.2412) [dB]
Inference Time: 0.9622387886047363
Extended Kalman Filter - MSE LOSS: tensor(-22.2893) [dB]
Extended Kalman Filter - STD: tensor(0.2891) [dB]
Inference Time: 0.963900089263916
Extended

### 2. REKF (fixed `c`, `use_nn=False`)

In [6]:
mses, times = [], []
rekf = RobustKalman(sys_model, test_inputs[0][0], c, False, False, sl_model=0)
for ti, tt in zip(test_inputs, test_targets):
    for s in range(ti.shape[0]):
        rekf.y = ti[s]
        t0 = time.time(); rekf.fnREKF(); dt = time.time() - t0
        mses.append(crit(rekf.Xn_out, tt[s]).item()); times.append(dt)
record('REKF', mses, times)

REKF              MSE(dB)= -18.633  STD(dB)= 0.188  t/seq= 104.90 ms  (nseq=10000)


### 3. KalmanNet

In [7]:
KNet = KalmanNetNN(); KNet.NNBuild(sys_model, args)
pipe = Pipeline_EKF('task4', 'KNet', 'KNet')
pipe.setssModel(sys_model); pipe.setModel(KNet); pipe.setTrainingParams(args)
pipe.NNTrain(sys_model, cv_input, cv_target, train_input, train_target, path_results)

mses, times = [], []
for ti, tt in zip(test_inputs, test_targets):
    t0 = time.time(); out = pipe.NNTest(sys_model, ti, tt, path_results); dt = time.time() - t0
    mses.extend(out[0].tolist()); times.extend([dt / ti.shape[0]] * ti.shape[0])
record('KalmanNet', mses, times)

Epoch 1/200, MSE Training: 0.0120, MSE Validation: 0.0091
Epoch 2/200, MSE Training: 0.0088, MSE Validation: 0.0071
Epoch 3/200, MSE Training: 0.0072, MSE Validation: 0.0057
Epoch 4/200, MSE Training: 0.0056, MSE Validation: 0.0049
Epoch 5/200, MSE Training: 0.0047, MSE Validation: 0.0044
Epoch 6/200, MSE Training: 0.0043, MSE Validation: 0.0041
Epoch 7/200, MSE Training: 0.0039, MSE Validation: 0.0039
Epoch 8/200, MSE Training: 0.0041, MSE Validation: 0.0039
Epoch 9/200, MSE Training: 0.0038, MSE Validation: 0.0038
Epoch 10/200, MSE Training: 0.0039, MSE Validation: 0.0038
Epoch 11/200, MSE Training: 0.0037, MSE Validation: 0.0038
Epoch 12/200, MSE Training: 0.0031, MSE Validation: 0.0037
Epoch 13/200, MSE Training: 0.0033, MSE Validation: 0.0036
Epoch 14/200, MSE Training: 0.0037, MSE Validation: 0.0034
Epoch 15/200, MSE Training: 0.0034, MSE Validation: 0.0033
Epoch 16/200, MSE Training: 0.0028, MSE Validation: 0.0031
Epoch 17/200, MSE Training: 0.0032, MSE Validation: 0.0029
Epoch 

### 4. Existing robust KalmanNet (professor's original)

Built exactly as the professor did (`hidden_layers=[20,20,20,20,20]`) via the
frozen original pair. **Untrained by construction**: a state-MSE `backward()`
gives `grad=None` on every parameter (verified), so training is a no-op and it
runs at its seeded initial `c`. One shared net is reused across all sequences.

In [8]:
existing_net = RobustKalmanOrig(sys_model, test_inputs[0][0], c, False, True,
                                input_feat_mode=3, hidden_layers=[20,20,20,20,20], sl_model=0).nn
mses, times = [], []
for ti, tt in zip(test_inputs, test_targets):
    for s in range(ti.shape[0]):
        rk = RobustKalmanOrig(sys_model, ti[s], c, False, True,
                              input_feat_mode=3, hidden_layers=[20,20,20,20,20], sl_model=0)
        rk.nn = existing_net
        rk.nn.previous_output = torch.zeros(1, 1)   # reset feedback per sequence
        with torch.no_grad():
            t0 = time.time(); rk.fnREKF(train=False); dt = time.time() - t0
        mses.append(crit(rk.Xn, tt[s]).item()); times.append(dt)
record('existing', mses, times)

existing          MSE(dB)= -18.707  STD(dB)= 0.190  t/seq= 163.82 ms  (nseq=10000)


### 5. Proposed method (GRU + BPTT)

Current `RobustKalman` (`use_nn=True`, GRU) trained with mini-batch BPTT on the
posterior state-MSE; `fcl_out` excluded from weight decay; gradient clipped.

In [9]:
prop = RobustKalman(sys_model, train_input[0], c, False, True,
                    input_feat_mode=3, gru_hidden_size=64, sl_model=0)
decay, no_decay = [], []
for pname, p_ in prop.nn.named_parameters():
    (no_decay if 'fcl_out' in pname else decay).append(p_)
opt = optim.Adam([{'params': decay, 'weight_decay': args.wd},
                  {'params': no_decay, 'weight_decay': 0.0}], lr=args.lr)

Npool = train_input.shape[0]
for epoch in range(args.n_steps):
    idx = random.sample(range(Npool), min(args.n_batch, Npool))
    opt.zero_grad(); losses = []
    for j in idx:
        prop.y = train_input[j]
        prop.fnREKF(train=True)
        losses.append(crit(prop.Xn_out, train_target[j]))
    loss = torch.stack(losses).mean()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(prop.nn.parameters(), 1.0)
    opt.step()
print(f'proposed: trained {args.n_steps} epochs, final loss={loss.item():.4f}')

mses, times = [], []
for ti, tt in zip(test_inputs, test_targets):
    for s in range(ti.shape[0]):
        prop.y = ti[s]
        with torch.no_grad():
            t0 = time.time(); prop.fnREKF(train=False); dt = time.time() - t0
        mses.append(crit(prop.Xn_out, tt[s]).item()); times.append(dt)
record('proposed', mses, times)

proposed: trained 200 epochs, final loss=0.0135
proposed          MSE(dB)= -18.707  STD(dB)= 0.190  t/seq= 178.80 ms  (nseq=10000)


## Results

In [10]:
order = ['EKF', 'REKF', 'KalmanNet', 'existing', 'proposed']
df = pd.DataFrame(results).T.reindex(order)
print(df.to_string(float_format=lambda x: f'{x:.4f}'))
df

           MSE [dB]  STD [dB]  t/seq [s]
EKF        -22.2118    0.2672     0.0097
REKF       -18.6329    0.1881     0.1049
KalmanNet  -28.2354    1.2591     0.0005
existing   -18.7065    0.1897     0.1638
proposed   -18.7068    0.1897     0.1788


,MSE [dB],STD [dB],t/seq [s]
EKF,-22.211822,0.267164,0.009726
REKF,-18.632895,0.188144,0.104897
KalmanNet,-28.235370,1.259108,0.000474
existing,-18.706541,0.189666,0.163822
proposed,-18.706790,0.189677,0.178797


In [11]:
fig = go.Figure()
fig.add_trace(go.Bar(x=order, y=[results[k]['MSE [dB]'] for k in order],
                     text=[f"{results[k]['MSE [dB]']:.2f}" for k in order],
                     textposition='outside'))
fig.update_layout(title=f'Task 4 — full-information posterior MSE [dB] ({CONFIG})',
                  yaxis_title='MSE [dB] (lower = better)', xaxis_title='estimator',
                  height=420, showlegend=False)
fig.show()

## Partial-information (model mismatch)

Filters run on a **mismatched** state model `f_partial` (KalmanNet-paper "Partial":
α=1, β=1, φ=0, δ=0 → f≈sin(x)); data still comes from the true "Full" `sys_model`.
KalmanNet is built on the partial model but **trained on the true data**, so it can
learn to compensate. Expected (the paper's headline): EKF/REKF/existing/proposed
degrade sharply; KalmanNet stays low. Robustness (`c`) cannot rescue this mismatch,
so REKF/existing/proposed ≈ EKF here.

In [12]:
def f_partial(x):
    return torch.sin(x)                       # alpha=1, beta=1, phi=0, delta=0

sys_p = SystemModel(f_partial, Q, h, R, args.T, args.T_test, m, n)
sys_p.InitSequence(m1x_0, m2x_0)
sys_p.alpha, sys_p.beta, sys_p.phi, sys_p.delta = 1.0, 1.0, 0.0, 0.0
sys_p.a, sys_p.b, sys_p.c = 1, 1, 0
sys_p.m1x_0 = torch.zeros(m, 1)               # same filter init as full-info

results_partial = {}
def record_p(name, mses, times):
    mdB = to_dB(float(np.mean(mses)))
    sdB = float(np.std([to_dB(x) for x in mses])); tavg = float(np.mean(times))
    results_partial[name] = {'MSE [dB]': mdB, 'STD [dB]': sdB, 't/seq [s]': tavg}
    print(f'{name:16s}  MSE(dB)={mdB:+8.3f}  STD(dB)={sdB:6.3f}  t/seq={tavg*1e3:7.2f} ms')

# EKF (mismatched)
mses, times = [], []
for ti, tt in zip(test_inputs, test_targets):
    t0 = time.time(); out = EKFTest(args, sys_p, ti, tt); dt = time.time() - t0
    mses.extend(out[0].tolist()); times.extend([dt / ti.shape[0]] * ti.shape[0])
record_p('EKF', mses, times)

# REKF (mismatched)
mses, times = [], []
rekf_p = RobustKalman(sys_p, test_inputs[0][0], c, False, False, sl_model=0)
for ti, tt in zip(test_inputs, test_targets):
    for s in range(ti.shape[0]):
        rekf_p.y = ti[s]; t0 = time.time(); rekf_p.fnREKF(); dt = time.time() - t0
        mses.append(crit(rekf_p.Xn_out, tt[s]).item()); times.append(dt)
record_p('REKF', mses, times)

# existing (mismatched; untrained shared net)
exnet_p = RobustKalmanOrig(sys_p, test_inputs[0][0], c, False, True,
                           input_feat_mode=3, hidden_layers=[20, 20, 20, 20, 20], sl_model=0).nn
mses, times = [], []
for ti, tt in zip(test_inputs, test_targets):
    for s in range(ti.shape[0]):
        rk = RobustKalmanOrig(sys_p, ti[s], c, False, True,
                              input_feat_mode=3, hidden_layers=[20, 20, 20, 20, 20], sl_model=0)
        rk.nn = exnet_p; rk.nn.previous_output = torch.zeros(1, 1)
        with torch.no_grad():
            t0 = time.time(); rk.fnREKF(train=False); dt = time.time() - t0
        mses.append(crit(rk.Xn, tt[s]).item()); times.append(dt)
record_p('existing', mses, times)

# proposed (mismatched; mini-batch BPTT)
prop_p = RobustKalman(sys_p, train_input[0], c, False, True,
                      input_feat_mode=3, gru_hidden_size=64, sl_model=0)
decay    = [p for nm, p in prop_p.nn.named_parameters() if 'fcl_out' not in nm]
no_decay = [p for nm, p in prop_p.nn.named_parameters() if 'fcl_out' in nm]
opt_p = optim.Adam([{'params': decay, 'weight_decay': args.wd},
                    {'params': no_decay, 'weight_decay': 0.0}], lr=args.lr)
for epoch in range(args.n_steps):
    idx = random.sample(range(train_input.shape[0]), min(args.n_batch, train_input.shape[0]))
    opt_p.zero_grad(); losses = []
    for j in idx:
        prop_p.y = train_input[j]; prop_p.fnREKF(train=True)
        losses.append(crit(prop_p.Xn_out, train_target[j]))
    loss = torch.stack(losses).mean(); loss.backward()
    torch.nn.utils.clip_grad_norm_(prop_p.nn.parameters(), 1.0); opt_p.step()
mses, times = [], []
for ti, tt in zip(test_inputs, test_targets):
    for s in range(ti.shape[0]):
        prop_p.y = ti[s]
        with torch.no_grad():
            t0 = time.time(); prop_p.fnREKF(train=False); dt = time.time() - t0
        mses.append(crit(prop_p.Xn_out, tt[s]).item()); times.append(dt)
record_p('proposed', mses, times)

# KalmanNet: built on the PARTIAL model, TRAINED on the TRUE data
KNet_p = KalmanNetNN(); KNet_p.NNBuild(sys_p, args)
pipe_p = Pipeline_EKF('task4-partial', 'KNet', 'KNet')
pipe_p.setssModel(sys_p); pipe_p.setModel(KNet_p); pipe_p.setTrainingParams(args)
pipe_p.NNTrain(sys_p, cv_input, cv_target, train_input, train_target, path_results)
mses, times = [], []
for ti, tt in zip(test_inputs, test_targets):
    t0 = time.time(); out = pipe_p.NNTest(sys_p, ti, tt, path_results); dt = time.time() - t0
    mses.extend(out[0].tolist()); times.extend([dt / ti.shape[0]] * ti.shape[0])
record_p('KalmanNet', mses, times)

Extended Kalman Filter - MSE LOSS: tensor(-1.2127) [dB]
Extended Kalman Filter - STD: tensor(0.0283) [dB]
Inference Time: 0.9209520816802979
Extended Kalman Filter - MSE LOSS: tensor(-1.2070) [dB]
Extended Kalman Filter - STD: tensor(0.0377) [dB]
Inference Time: 0.8563339710235596
Extended Kalman Filter - MSE LOSS: tensor(-1.2112) [dB]
Extended Kalman Filter - STD: tensor(0.0329) [dB]
Inference Time: 0.867542028427124
Extended Kalman Filter - MSE LOSS: tensor(-1.2066) [dB]
Extended Kalman Filter - STD: tensor(0.0314) [dB]
Inference Time: 0.8663449287414551
Extended Kalman Filter - MSE LOSS: tensor(-1.2065) [dB]
Extended Kalman Filter - STD: tensor(0.0326) [dB]
Inference Time: 0.867847204208374
Extended Kalman Filter - MSE LOSS: tensor(-1.2162) [dB]
Extended Kalman Filter - STD: tensor(0.0324) [dB]
Inference Time: 1.198193073272705
Extended Kalman Filter - MSE LOSS: tensor(-1.2077) [dB]
Extended Kalman Filter - STD: tensor(0.0302) [dB]
Inference Time: 1.0016098022460938
Extended Kalman 

In [13]:
order = ['EKF', 'REKF', 'KalmanNet', 'existing', 'proposed']
combined = pd.DataFrame({
    'MSE dB [Full]':    [results[k]['MSE [dB]']         for k in order],
    'MSE dB [Partial]': [results_partial[k]['MSE [dB]'] for k in order],
}, index=order)
print(combined.to_string(float_format=lambda x: f'{x:.3f}'))

fig = go.Figure()
fig.add_trace(go.Bar(name='Full',    x=order, y=[results[k]['MSE [dB]']         for k in order]))
fig.add_trace(go.Bar(name='Partial', x=order, y=[results_partial[k]['MSE [dB]'] for k in order]))
fig.update_layout(barmode='group', title='Task 4 — MSE [dB]: full vs partial information',
                  yaxis_title='MSE [dB] (lower = better)', xaxis_title='estimator', height=440)
fig.show()
combined

           MSE dB [Full]  MSE dB [Partial]
EKF              -22.212            -1.208
REKF             -18.633            -1.208
KalmanNet        -28.235           -20.875
existing         -18.707            -1.208
proposed         -18.707            -1.208


,MSE dB [Full],MSE dB [Partial]
EKF,-22.211822,-1.208451
REKF,-18.632895,-1.208451
KalmanNet,-28.235370,-20.874885
existing,-18.706541,-1.208451
proposed,-18.706790,-1.208451


## Reading the result (full information)

On the **matched** model, covariance inflation is a *penalty* (nothing to be
robust against), so **EKF ≤ REKF ≈ existing ≈ proposed**, while **KalmanNet**
leads by learning the gain directly. `existing ≈ REKF` (the original's `c` is
untrained and, even so, `c` has little leverage here — a flat MSE-vs-`c`
landscape). The proposed method's advantage is expected only under **partial
information / model mismatch**, which is a separate build once `c_range` is
widened. See `docs/ORIGINAL_VS_MODIFIED.md`.